In [ ]:
import sys
import os
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

load_dotenv(override=True)

import os
from pathlib import Path
from typing import Optional
from typing import List

def get_file_path(file_path: str) -> str:
    """
    파일 경로를 절대 경로로 변환하는 함수
    
    Args:
        file_path: 상대 경로 또는 절대 경로
    """
    # 파일 경로 확인 및 절대 경로로 변환
    file_path = Path(file_path)
    if not file_path.is_absolute():
        # 노트북 위치 기준 상대 경로 처리
        # 노트북은 asset_ai_portal/tests 폴더에 있고, documents는 20_code_test 루트에 있음
        current_dir = Path.cwd()
        
        # asset_ai_portal/tests에서 실행 중이면 상위로 두 번 이동 (20_code_test 루트)
        if current_dir.name == 'tests' and current_dir.parent.name == 'asset_ai_portal':
            project_root = current_dir.parent.parent  # tests -> asset_ai_portal -> 20_code_test
        elif current_dir.name == 'asset_ai_portal':
            project_root = current_dir.parent  # asset_ai_portal -> 20_code_test
        else:
            # 20_code_test에서 실행 중이면 그대로 사용
            project_root = current_dir
        
        file_path = project_root / file_path
    
    if not file_path.exists():
        raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {file_path}")
    
    return str(file_path)


def table_to_markdown(table: List[List]) -> str:
    """
    표 데이터를 마크다운 테이블 형식으로 변환하는 헬퍼 함수
    
    Args:
        table: 2차원 리스트 형태의 표 데이터
    
    Returns:
        마크다운 테이블 문자열
    """
    if not table or len(table) == 0:
        return ""
    
    # 빈 셀을 빈 문자열로 변환
    def clean_cell(cell):
        if cell is None:
            return ""
        return str(cell).strip()
    
    # 표 데이터 정리
    cleaned_table = [[clean_cell(cell) for cell in row] for row in table]
    
    # 최대 컬럼 수 확인
    max_cols = max(len(row) for row in cleaned_table) if cleaned_table else 0
    
    # 모든 행을 동일한 컬럼 수로 맞춤
    normalized_table = []
    for row in cleaned_table:
        normalized_row = row + [""] * (max_cols - len(row))
        normalized_table.append(normalized_row)
    
    if not normalized_table:
        return ""
    
    markdown_lines = []
    
    # 헤더 행 (첫 번째 행)
    header = normalized_table[0]
    markdown_lines.append("| " + " | ".join(header) + " |")
    
    # 구분선
    markdown_lines.append("| " + " | ".join(["---"] * len(header)) + " |")
    
    # 데이터 행들
    for row in normalized_table[1:]:
        markdown_lines.append("| " + " | ".join(row) + " |")
    
    return "\n".join(markdown_lines)

# pdfplumber

In [ ]:
def extract_text_from_pdf_with_pdfplumber(pdf_path: str, password: str = None) -> str:
    """
    pdfplumber를 사용하여 PDF 파일을 마크다운 형식으로 변환하는 함수
    
    pdfplumber는 PDF 파일을 텍스트 데이터로 추출하는 라이브러리로, 표, 이미지, 레이아웃 등을 잘 보존합니다.
    암호화된 PDF와 암호화되지 않은 PDF 모두 처리할 수 있습니다.
    """
    try:
        import pdfplumber
    except ImportError:
        raise ImportError(
            "PDF를 처리하기 위해 pdfplumber가 필요합니다.\n"
            "설치 명령: pip install pdfplumber"
        )
    
    markdown_parts = []
    
    try:
        pdf_path = get_file_path(pdf_path)
        # password가 있으면 암호화된 PDF로 처리, 없으면 암호화되지 않은 PDF로 처리
        pdf_kwargs = {"password": password} if password else {}
        
        with pdfplumber.open(pdf_path, **pdf_kwargs) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                page_content = []
                
                # 표 추출 (표가 있으면 먼저 표를 추출)
                tables = page.extract_tables()
                if tables:
                    for table_idx, table in enumerate(tables):
                        if table:
                            markdown_table = table_to_markdown(table)
                            if markdown_table:
                                page_content.append(markdown_table)
                                page_content.append("")  # 표 다음에 빈 줄 추가
                
                # 텍스트 추출
                text = page.extract_text()
                if text:
                    page_content.append(text)
                
                if page_content:
                    markdown_parts.append("\n".join(page_content))
        
        return "\n\n".join(markdown_parts) if markdown_parts else ""
        
    except Exception as e:
        # 암호화 관련 오류인지 확인
        error_msg = str(e).lower()
        if 'password' in error_msg or 'encrypted' in error_msg or 'incorrect password' in error_msg:
            raise ValueError(f"PDF 암호가 올바르지 않거나 암호화된 PDF를 읽을 수 없습니다: {e}")
        raise

# document load

In [ ]:
# Document Loader - 파일 형식에 따라 적절한 함수 호출

from pathlib import Path
from typing import Union

def load_document(file_path: str, password: str = None) -> str:
    """
    파일 형식에 따라 적절한 텍스트 추출 함수를 호출하여 텍스트를 반환하는 통합 함수
    
    지원 형식:
    - PDF: .pdf 파일 (암호화된 PDF 지원)
    - Excel: .xlsx, .xls 파일
    
    Args:
        file_path: 문서 파일 경로 (상대 경로 또는 절대 경로)
        password: PDF 파일이 암호화된 경우 비밀번호 (선택사항)
    
    Returns:
        추출된 텍스트 문자열
        - PDF: 전체 텍스트
        - Excel: 모든 시트의 텍스트를 합친 문자열
    
    Raises:
        FileNotFoundError: 파일을 찾을 수 없을 때
        ValueError: 지원하지 않는 파일 형식일 때 또는 PDF 암호가 틀렸을 때
        ImportError: 필요한 라이브러리가 설치되지 않았을 때
    """
    file_path_obj = Path(file_path)
    file_ext = file_path_obj.suffix.lower()
    
    # 파일 형식에 따라 적절한 함수 호출
    if file_ext == '.pdf':
        # PDF 파일 처리 (암호 전달)
        # return extract_text_from_pdf(file_path, password=password)
        return extract_text_from_pdf_with_pdfplumber(file_path, password)
    
    # elif file_ext in ['.xlsx', '.xls']:
    #     # Excel 파일 처리 - 모든 시트의 텍스트를 하나의 문자열로 반환
    #     return get_all_sheets_text(file_path)
    
    else:
        raise ValueError(
            f"지원하지 않는 파일 형식입니다: {file_ext}\n"
            f"지원 형식: .pdf, .xlsx, .xls"
        )

In [ ]:
# text 추출

# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx"
_document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(2차)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127_20260105_182813.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/카디프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/하나생명(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/iM라이프_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"

_password = None
_password = '345678'

document_text = load_document(_document_file_path, _password)
print(document_text)

# LLM 문서 보정

In [ ]:
# 문서가 변액일임펀드 설정/해지 지시서인지 확인하는 LLM 노드 생성

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
# LLM 모델 정의

LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")
LLM_TEMPERATURE=os.getenv("LLM_TEMPERATURE")

# vLLM 모델 인스턴스 생성
llm = init_chat_model(
    "openai:",
    temperature=LLM_TEMPERATURE,
    top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.9로 설정
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY
)

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

_gettering_data = None
if document_text:
    # 메시지 객체 생성
    system_msg = SystemMessage("당신은 자산운용사에서 변액일임펀드 설정/해지 업무 담당자 입니다.")
    human_msg = HumanMessage(f"""
    아래는 수익자가 보내온 변액일임펀드 설정/해지 지시서 PDF 파일에서 pdfplumber 라이브러리를 사용하여 추출한 data입니다.
    추출 data는 table을 markdown 형식으로 변환하여 추출한 data와 텍스트 형식으로 추출한 data로 구성되어 있습니다.
    지침에 따라 추출 data를 정리하세요.
    
    ### 변액일임펀드 설정/해지 지시서 내용 ###
    {document_text}

    ** 반드시 지켜야 할 중요 지침 **
    1. 전체 내용을 분석하세요.
    2. 전체 내용을 markdown 형식으로 수정하세요.
    3. markdown table 코드를 오류가 없는 정상적인 코드로 수정하세요.    
    4. 추측과 예상 또는 설명 등의 첨언은 하지 말고 추출 data 내용만 출력하세요.
    5. 중복되는 내용은 제거하세요.
    6. 정리 결과에서 누락된 데이터가 있는지 확인하세요. 누락된 데이터가 있으면 추가하세요.
    7. 정리 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
    8. 정리 결과의 오류 여부를 검증하고 오류가 있으면 수정하세요.    
    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]
    response = llm.invoke(messages)  # AIMessage 반환
    # print(response)
    _gettering_data = response.content

In [ ]:
from IPython.display import Markdown, display

def display_markdown(response):
    # LLM 응답을 마크다운 형식으로 보기 좋게 표시
    if 'response' in locals():
        display(Markdown(response.content))
        
        # 추가 정보 (토큰 사용량 등)를 표시
        if hasattr(response, 'response_metadata') and response.response_metadata:
            metadata = response.response_metadata
            if 'token_usage' in metadata:
                print("\n---")
                print("**토큰 사용량:**")
                print(f"- 입력 토큰: {metadata['token_usage'].get('prompt_tokens', 'N/A')}")
                print(f"- 출력 토큰: {metadata['token_usage'].get('completion_tokens', 'N/A')}")
                print(f"- 총 토큰: {metadata['token_usage'].get('total_tokens', 'N/A')}")
    else:
        print("⚠️ 'response' 변수를 찾을 수 없습니다. 먼저 LLM을 호출해주세요.")

display_markdown(response)